> **Experiment D: Local per-turbine, no collaboration**
> Spec: [Arm D](../docs/experiments/D_local_per_turbine.md) · Depends on E03
> Config: [arm_local_turbine.yaml](../configs/arm_local_turbine.yaml)
> Registered in `experiments/registry.yaml`.

# Arm D: train from the shared client artifacts
This is the local baseline for federated learning. It consumes E03's saved arrays exactly: same memberships, ordered features, training-only scaler, and status-only evaluation mask. The old notebook outputs were cleared because this preprocessing change requires a fresh run.

On Kaggle attach the bridge source ZIP (or source checkout) and the E03 output dataset. Set `WT_CLIENT_DATASET` to the folder with `dataset_manifest.json`. Training requires TensorFlow; output needs pyarrow for score export. This notebook never edits the attached dataset. Set `WT_OUTPUT_ROOT` to change the run output location. Use GPU for training if available. No TensorFlow training was run as part of the code review.

## 1. Import the shared source

In [ ]:
import os, sys, json, zipfile, subprocess, hashlib
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd

SMOKE = os.environ.get("WT_CLIENT_SMOKE", "0") == "1"
WORK_ROOT = Path(os.environ.get("WT_WORK_ROOT", "/kaggle/working" if Path("/kaggle").exists() else str(Path.cwd())))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
MODULE = Path("src/windturbine/data/clients.py")
explicit_root, requested_ref = os.environ.get("WT_REPO_ROOT"), os.environ.get("WT_REPO_REF")
if explicit_root and requested_ref:
    raise ValueError("Choose WT_REPO_ROOT or WT_REPO_REF, not both")
REPO, bundle_sha256, resolved_ref = None, None, None
if explicit_root:
    REPO = Path(explicit_root)
    if not (REPO / MODULE).is_file():
        raise RuntimeError("WT_REPO_ROOT does not contain the client bridge: " + str(REPO))
elif requested_ref:
    REPO = WORK_ROOT / "windturbine-client-bridge-git"
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/PranavMundhra/Wind-Turbine.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", requested_ref], check=True)
    resolved_ref = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
    expected_ref = subprocess.run(["git", "-C", str(REPO), "rev-parse", requested_ref + "^{commit}"], check=True, capture_output=True, text=True).stdout.strip()
    assert resolved_ref == expected_ref, "Requested source revision was not selected"
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    if Path("/kaggle/input").exists():
        candidates += [p.parent.parent.parent.parent for p in Path("/kaggle/input").rglob("src/windturbine/data/clients.py")]
    REPO = next((p for p in candidates if (p / MODULE).is_file()), None)
    if REPO is None and Path("/kaggle/input").exists():
        bundles = list(Path("/kaggle/input").rglob("windturbine-client-bridge.zip"))
        if len(bundles) > 1:
            raise RuntimeError("Attach only one source ZIP, or set WT_REPO_ROOT")
        if bundles:
            bundle_sha256 = hashlib.sha256(bundles[0].read_bytes()).hexdigest()
            target = WORK_ROOT / ("windturbine-source-" + bundle_sha256[:16])
            with zipfile.ZipFile(bundles[0]) as archive:
                names = archive.namelist()
                if len(names) != len(set(names)):
                    raise ValueError("Duplicate paths in source archive")
                for name in names:
                    if "\\" in name or not (target / name).resolve().is_relative_to(target.resolve()):
                        raise ValueError("Unsafe source archive path")
                package_manifest = json.loads(archive.read("bridge_bundle_manifest.json"))
                if package_manifest.get("format_version") != 1 or not isinstance(package_manifest.get("files"), dict):
                    raise ValueError("Unsupported source bundle manifest")
                files = package_manifest["files"]
                if set(names) != set(files) | {"bridge_bundle_manifest.json"}:
                    raise ValueError("Source bundle file list does not match manifest")
                for name, digest in files.items():
                    if hashlib.sha256(archive.read(name)).hexdigest() != digest:
                        raise ValueError("Source bundle hash mismatch: " + name)
                archive.extractall(target)
            REPO = target
if REPO is None or not (REPO / MODULE).is_file():
    raise RuntimeError("Attach windturbine-client-bridge.zip as a Kaggle Dataset (internet off works), "
                       "or attach a source checkout and set WT_REPO_ROOT. Optional internet mode: "
                       "set WT_REPO_REF to a published commit/tag containing src/windturbine/data/clients.py.")
# Also verify an already extracted source bundle before importing any package code.
if (REPO / "bridge_bundle_manifest.json").is_file():
    package_manifest = json.loads((REPO / "bridge_bundle_manifest.json").read_text(encoding="utf-8"))
    if package_manifest.get("format_version") != 1:
        raise ValueError("Unsupported extracted source bundle")
    for name, digest in package_manifest["files"].items():
        source_path = (REPO / name).resolve()
        if not source_path.is_relative_to(REPO.resolve()) or hashlib.sha256(source_path.read_bytes()).hexdigest() != digest:
            raise ValueError("Extracted source bundle hash mismatch: " + name)
source_notebooks = [REPO / "notebooks" / name for name in ("13_arm_local_turbine.ipynb", "20_federated_clients.ipynb")]
source_files = sorted([*REPO.glob("src/windturbine/**/*.py"), *REPO.glob("configs/*.yaml"),
                       *(p for p in source_notebooks if p.is_file())])
source_hashes = {p.relative_to(REPO).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}
SOURCE_IDENTITY = {"source_tree_sha256": hashlib.sha256(json.dumps(source_hashes, sort_keys=True).encode()).hexdigest(),
                   "source_files": source_hashes, "bundle_sha256": bundle_sha256,
                   "requested_git_ref": requested_ref, "resolved_git_commit": resolved_ref}
sys.path.insert(0, str(REPO / "src"))
from windturbine.config import load_config
from windturbine.data.clients import build_client_dataset, load_client_dataset
OUTPUT_ROOT = Path(os.environ.get("WT_CLIENT_OUTPUT_ROOT", os.environ.get("WT_CLIENT_OUTPUT", str(WORK_ROOT / "windturbine_clients"))))
if Path("/kaggle").exists() and not OUTPUT_ROOT.resolve().is_relative_to(Path("/kaggle/working").resolve()):
    raise ValueError("On Kaggle, WT_CLIENT_OUTPUT_ROOT must be inside /kaggle/working")
print("Source:", REPO, "| Output:", OUTPUT_ROOT)
print("Source identity:", json.dumps(SOURCE_IDENTITY, indent=2))

## 2. Locate the saved E03 dataset and load configuration

In [ ]:
from windturbine.runs import (new_run_dir, set_seeds, write_manifest, write_results,
                              write_scores, from_evaluate_care, file_hash)
from windturbine.scoring import evaluate_care
CFG = load_config(REPO / "configs/base.yaml", REPO / "configs/arm_local_turbine.yaml")
CFG = replace(CFG, seed=int(os.environ.get("WT_SEED", CFG.seed)))
if os.environ.get("WT_CLIENT_DATASET"):
    CLIENT_ROOT = Path(os.environ["WT_CLIENT_DATASET"])
elif (OUTPUT_ROOT / "dataset_manifest.json").exists():
    CLIENT_ROOT = OUTPUT_ROOT
else:
    found = list(Path("/kaggle/input").rglob("dataset_manifest.json")) if Path("/kaggle/input").exists() else []
    if len(found) != 1:
        raise RuntimeError("Run E03 first, attach its output and set WT_CLIENT_DATASET to the folder with dataset_manifest.json.")
    CLIENT_ROOT = found[0].parent
manifest_path = CLIENT_ROOT / "dataset_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert len(manifest["clients"]) == 22 and manifest["n_events"] == 58, "Arm D requires the complete real dataset, not smoke data"
print("Saved E03:", CLIENT_ROOT, "| Dataset manifest SHA256:", file_hash(manifest_path))
print("Saved preprocessing:", manifest["preprocessing_config"])
# Data/filter/split settings must match the E03 build. Model seed can differ.
from windturbine.data.clients import preprocessing_config
assert manifest["preprocessing_config"] == preprocessing_config(replace(CFG, seed=manifest["preprocessing_config"]["seed"])), "D configuration differs from the saved E03 preprocessing; rebuild E03 or use its exact data settings"
RUN_ROOT = Path(os.environ.get("WT_OUTPUT_ROOT", str(WORK_ROOT / CFG.output_dir)))
if Path("/kaggle").exists() and not RUN_ROOT.resolve().is_relative_to(Path("/kaggle/working").resolve()):
    raise ValueError("On Kaggle, run outputs must be inside /kaggle/working")
SEEDING = set_seeds(CFG.seed)
RUN_DIR = new_run_dir("D", CFG, root=RUN_ROOT)
write_manifest(RUN_DIR, "D", CFG, upstream={"E03": file_hash(manifest_path)},
               data_manifest=manifest_path, seeding=SEEDING, repo_root=REPO)
run_manifest_path = RUN_DIR / "manifest.json"
run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
run_manifest["source_identity"] = SOURCE_IDENTITY
run_manifest_path.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")


## 3. Model and training
The dense autoencoder, Gaussian noise option, Adam optimizer, early stopping and learning-rate schedule are preserved. Features are already scaled by E03; do not fit another scaler. Locally inactive feature dimensions are explicitly zero-filled. Training loss and reconstruction L2 include every shared dimension, so future FL models must use the same convention. Model seeds may differ while the saved E03 split seed stays fixed.

In [ ]:
# =============================================================================
# MODEL  --  paper Table 3 architecture, Wind Farm C
# =============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
tf.get_logger().setLevel("ERROR")
tf.random.set_seed(CFG.seed)

def build_autoencoder(n_features, hidden=CFG.hidden_layers, noise=CFG.noise_std, lr=CFG.learning_rate):
    inp = layers.Input(shape=(n_features,))
    x = layers.GaussianNoise(noise)(inp) if noise > 0 else inp
    names = ("encode1", "encode2", "bottleneck", "decode1", "decode2") if len(hidden) == 5 else tuple(f"hidden_{i}" for i in range(len(hidden)))
    for name, units in zip(names, hidden):
        x = layers.Dense(units, activation="relu", name=name)(x)
    out = layers.Dense(n_features, activation="linear", name="output")(x)
    m = models.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(learning_rate=lr), loss="mse")
    return m

def fit_autoencoder(Xtr, Xva, verbose=0, tag=""):
    tf.keras.backend.clear_session()
    tf.random.set_seed(CFG.seed)
    model = build_autoencoder(Xtr.shape[1])
    cbs = [callbacks.EarlyStopping(monitor="val_loss", patience=CFG.early_stop_patience,
                                   restore_best_weights=True, verbose=0)]
    if CFG.use_lr_plateau:
        cbs.append(callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=CFG.lr_plateau_factor,
            patience=CFG.lr_plateau_patience, min_lr=1e-6, verbose=0))
    hist = model.fit(Xtr, Xtr, validation_data=(Xva, Xva),
                     batch_size=CFG.batch_size, epochs=CFG.max_epochs,
                     callbacks=cbs, verbose=verbose, shuffle=True)
    tr_hist = hist.history["loss"]
    info = dict(tag=tag, epochs_run=len(tr_hist),
                best_val=float(np.min(hist.history["val_loss"])),
                final_train=float(tr_hist[-1]),
                # >0 means training loss ended HIGHER than it started: divergence
                train_drift=float(tr_hist[-1] - min(tr_hist)))
    return model, info

In [ ]:
def reconstruction_error(model, X, batch_size=8192):
    out = np.empty(len(X), dtype=np.float32)
    for start in range(0, len(X), batch_size):
        batch = X[start:start + batch_size]
        out[start:start + len(batch)] = np.linalg.norm(batch - model(batch, training=False).numpy(), axis=1)
    return out

## 4. Train each client locally and score its prediction rows

In [ ]:
import gc, time, marshal
prediction_records, dataset_meta, fit_log = {}, {}, []
t0 = time.time()
for aid in manifest["clients"]:
    client = load_client_dataset(CLIENT_ROOT, aid, verify=True)
    assert client["schema"] == manifest["schema"], "Feature order differs"
    Xtr, Xva = client["train"]["X"], client["val"]["X"]
    assert len(Xtr) and len(Xva)
    model, info = fit_autoencoder(Xtr, Xva, tag=str(aid))
    model_contract = {"schema_hash": manifest["schema_hash"], "ordered_features": client["schema"],
                      "executed_function_sha256": {fn.__name__: hashlib.sha256(marshal.dumps(fn.__code__)).hexdigest()
                                                    for fn in (build_autoencoder, fit_autoencoder, reconstruction_error)},
                      "layers": [{"name": layer.name, "weight_shapes": [list(w.shape) for w in layer.get_weights()]}
                                 for layer in model.layers if layer.get_weights()]}
    contract_path = RUN_DIR / "model_contract.json"
    if contract_path.exists():
        assert json.loads(contract_path.read_text(encoding="utf-8")) == model_contract
    else:
        contract_path.write_text(json.dumps(model_contract, indent=2), encoding="utf-8")
    va_scores = reconstruction_error(model, Xva)
    threshold = float(np.quantile(va_scores, CFG.score_quantile))
    median = float(np.median(va_scores))
    if threshold / max(median, 1e-9) > 10:
        print("WARNING: high threshold/median ratio for turbine", aid)
    info.update(asset_id=aid, n_train=len(Xtr), n_val=len(Xva), threshold=threshold,
                val_med=median, val_p99=float(np.quantile(va_scores, 0.99)))
    fit_log.append(info)
    prediction = client["prediction"]
    for event in client["events"]:
        eid = event["event_id"]
        selected = prediction["event_id"].astype(str) == str(eid)
        assert selected.any(), f"Event {eid} has no prediction rows"
        scores = reconstruction_error(model, prediction["X"][selected])
        dataset_meta[eid] = dict(event, event_start=pd.to_datetime(event["event_start"]),
                                 event_end=pd.to_datetime(event["event_end"]))
        prediction_records[eid] = {"time_stamp": prediction["time_stamp"][selected],
                                   "normal_mask": prediction["eval_mask"][selected],
                                   "scores": scores, "prediction": (scores > threshold).astype(int)}
    print("Turbine", aid, "| train", len(Xtr), "| validation", len(Xva), "| threshold", threshold)
    del client, model, Xtr, Xva
    tf.keras.backend.clear_session()
    gc.collect()
assert len(prediction_records) == manifest["n_events"]
print("Training minutes:", (time.time() - t0) / 60)

## 5. Evaluate and save traceable results
CARE uses the shared evaluator. Its evaluation mask excludes abnormal status only. The result records the exact E03 manifest hash, source checkout and model configuration. Compare future FL runs only with D runs using the same E03 manifest and scoring version.

In [ ]:
results, per_event = evaluate_care(dataset_meta, prediction_records, beta=CFG.beta,
                                  criticality_threshold=CFG.criticality_threshold,
                                  care_weights=CFG.care_weights)
results["arm"] = CFG.arm
results["client_dataset_manifest_sha256"] = file_hash(manifest_path)
results["client_schema_hash"] = manifest["schema_hash"]
results["preprocessing_hash"] = manifest["preprocessing_hash"]
write_results(RUN_DIR, *from_evaluate_care(results, per_event))
write_scores(RUN_DIR, prediction_records)
pd.DataFrame(fit_log).to_csv(RUN_DIR / "fit_log.csv", index=False)
print(per_event.to_string(index=False))
print("Saved:", RUN_DIR)